NOTEBOOK 02: YOLOv11m TRAINING ON BALANCED DATASET

Kaggle Notebook — Retail Shelf Detection

This notebook:
  1. Loads the balanced dataset from Notebook 01
  2. Trains YOLOv11m at 1280×1280 with enhanced augmentation
  3. Validates on the test set
  4. Compares against all previous experiments
  5. Exports weights + results for deployment

Prerequisites:
  - Run Notebook 01 first OR upload balanced_dataset as a Kaggle dataset
  - GPU accelerator enabled (P100 or T4)


## CELL 1: SETUP


In [ ]:
!pip install ultralytics>=8.3.0 -q

import yaml
import json
import torch
from pathlib import Path
from ultralytics import YOLO

WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = str(WORKING_DIR / "runs")

# Check GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ── Dataset paths ──
# Option A: If Notebook 01 ran in the same session
BALANCED_DIR = WORKING_DIR / "balanced_dataset"

# Option B: If balanced dataset was uploaded as a separate Kaggle dataset
if not BALANCED_DIR.exists():
    BALANCED_DIR = Path("/kaggle/input/balanced-retail-shelf-dataset")

# Option C: Use original dataset (if no rebalancing done)
if not BALANCED_DIR.exists():
    BALANCED_DIR = Path("/kaggle/input/retail-shelf-dataset")
    print("⚠️  Using original (unbalanced) dataset")

print(f"\n📁 Dataset: {BALANCED_DIR}")
for split in ['train', 'valid', 'test']:
    imgs = list((BALANCED_DIR / split / "images").glob("*.jpg"))
    print(f"   {split}: {len(imgs)} images")

## CELL 2: CONFIGURE DATA.YAML


In [ ]:
# Create data.yaml with absolute paths
with open(BALANCED_DIR / "data.yaml", 'r') as f:
    cfg = yaml.safe_load(f)

cfg['train'] = str(BALANCED_DIR / "train" / "images")
cfg['val'] = str(BALANCED_DIR / "valid" / "images")
cfg['test'] = str(BALANCED_DIR / "test" / "images")

DATA_YAML = WORKING_DIR / "data.yaml"
with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"✅ data.yaml configured:")
print(f"   Train: {cfg['train']}")
print(f"   Val:   {cfg['val']}")
print(f"   Test:  {cfg['test']}")
print(f"   Classes: {cfg['nc']}")

## CELL 3: TRAIN YOLOv11m @ 1280×1280


In [ ]:
print("=" * 65)
print("  TRAINING: YOLOv11m @ 1280×1280 + Enhanced Augmentation")
print("  on Balanced Dataset (resampled minority classes)")
print("=" * 65)

model = YOLO("yolo11m.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=1280,
    batch=8,
    device=0,
    project=OUTPUT_DIR,
    name="yolov11m_balanced",
    exist_ok=True,
    verbose=True,

    # ── Optimized hyperparameters ──
    lr0=0.005,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    # ── Enhanced augmentation ──
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    degrees=15.0,
    scale=0.7,
    shear=5.0,
    flipud=0.1,
    fliplr=0.5,
    translate=0.2,
    perspective=0.001,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
)

print("\n✅ Training complete!")

## CELL 4: VALIDATE ON TEST SET


In [ ]:
print("=" * 65)
print("  VALIDATION ON TEST SET")
print("=" * 65)

metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=1280,
    device=0,
    project=OUTPUT_DIR,
    name="yolov11m_balanced_test",
    exist_ok=True,
)

p = metrics.box.mp
r = metrics.box.mr
m50 = metrics.box.map50
m5095 = metrics.box.map

print(f"\n{'='*60}")
print(f"  YOLOv11m BALANCED DATASET — TEST RESULTS")
print(f"{'='*60}")
print(f"  Precision:   {p*100:.2f}%")
print(f"  Recall:      {r*100:.2f}%")
print(f"  mAP@50:      {m50*100:.2f}%")
print(f"  mAP@50-95:   {m5095*100:.2f}%")
print(f"{'='*60}")

## CELL 5: COMPARE AGAINST ALL PREVIOUS EXPERIMENTS


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# All previous experiments
experiments = [
    {"model": "YOLOv11n",     "imgsz": 640,  "precision": 74.87, "recall": 66.47, "map50": 76.87},
    {"model": "v11n+LowConf", "imgsz": 640,  "precision": 67.84, "recall": 66.90, "map50": 69.84},
    {"model": "v11n+Aug",     "imgsz": 640,  "precision": 76.88, "recall": 68.78, "map50": 79.21},
    {"model": "YOLOv11m",     "imgsz": 640,  "precision": 80.99, "recall": 76.58, "map50": 84.52},
    {"model": "v11m@1280",    "imgsz": 1280, "precision": 72.68, "recall": 85.62, "map50": None},
    {"model": "RT-DETR-L",    "imgsz": 640,  "precision": 65.55, "recall": 85.76, "map50": 88.36},
    {"model": "RT-DETR-X",    "imgsz": 640,  "precision": 63.67, "recall": 85.56, "map50": 86.93},
    {"model": "v11m+Balanced","imgsz": 1280, "precision": p*100, "recall": r*100, "map50": m50*100},
]

# Print comparison table
print(f"\n{'='*80}")
print(f"  ALL EXPERIMENTS — COMPARISON")
print(f"{'='*80}")
print(f"  {'Model':<16} {'ImgSz':>6} {'Precision':>10} {'Recall':>8} {'mAP@50':>8}")
print(f"  {'-'*58}")
for exp in experiments:
    p_str = f"{exp['precision']:.2f}%" if exp['precision'] else "—"
    r_str = f"{exp['recall']:.2f}%" if exp['recall'] else "—"
    m_str = f"{exp['map50']:.2f}%" if exp['map50'] else "—"
    marker = " ⭐" if exp['model'] == 'v11m+Balanced' else ""
    print(f"  {exp['model']:<16} {exp['imgsz']:>6} {p_str:>10} {r_str:>8} {m_str:>8}{marker}")
print(f"{'='*80}")

# Precision vs Recall comparison chart
fig, ax = plt.subplots(figsize=(14, 6))
labels = [e['model'] for e in experiments]
prec = [e['precision'] for e in experiments]
rec = [e['recall'] for e in experiments]
x = np.arange(len(labels))

bars1 = ax.bar(x - 0.2, prec, 0.35, label='Precision', color='#3498db', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + 0.2, rec, 0.35, label='Recall', color='#2ecc71', edgecolor='white', linewidth=0.5)

# Highlight the latest result
bars1[-1].set_color('#1a5276')
bars1[-1].set_edgecolor('#3498db')
bars1[-1].set_linewidth(2)
bars2[-1].set_color('#196f3d')
bars2[-1].set_edgecolor('#2ecc71')
bars2[-1].set_linewidth(2)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Precision vs Recall — All Experiments', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=10)
ax.set_ylim(50, 100)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(WORKING_DIR / "experiment_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Comparison chart saved!")

## CELL 6: EXPORT WEIGHTS & RESULTS


In [ ]:
import shutil
from IPython.display import FileLink

# Save experiment result as JSON
result_data = {
    "model": "YOLOv11m",
    "imgsz": 1280,
    "dataset": "balanced",
    "augmentation": "enhanced",
    "precision": round(p * 100, 2),
    "recall": round(r * 100, 2),
    "map50": round(m50 * 100, 2),
    "map50_95": round(m5095 * 100, 2),
}
with open(WORKING_DIR / "balanced_result.json", 'w') as f:
    json.dump(result_data, f, indent=2)

# Zip weights + results
run_dir = Path(OUTPUT_DIR) / "yolov11m_balanced"
weights_dir = run_dir / "weights"

output_zip_dir = WORKING_DIR / "export"
output_zip_dir.mkdir(exist_ok=True)

# Copy key files
if weights_dir.exists():
    shutil.copy2(weights_dir / "best.pt", output_zip_dir / "best.pt")
    shutil.copy2(weights_dir / "last.pt", output_zip_dir / "last.pt")
shutil.copy2(WORKING_DIR / "balanced_result.json", output_zip_dir / "result.json")

# Training curves
for f in run_dir.glob("*.png"):
    shutil.copy2(f, output_zip_dir / f.name)

shutil.make_archive(str(WORKING_DIR / "yolov11m_balanced_export"), 'zip', str(output_zip_dir))

print("✅ Export complete!")
print(f"\n📦 Contains:")
for f in sorted(output_zip_dir.iterdir()):
    size = f.stat().st_size / 1024
    print(f"   {f.name} ({size:.0f} KB)")

print("\n📥 Download:")
display(FileLink("yolov11m_balanced_export.zip"))

print(f"\n{'='*60}")
print(f"  NEXT STEP:")
print(f"  Download best.pt and place it in your project:")
print(f"  retail-shelf-detection/models/best.pt")
print(f"  Then run: docker compose up --build")
print(f"{'='*60}")